# Catalog Visual Analysis

This notebook is read-only with respect to catalogs. The authoritative
7DT-FINALCAT-SDSS catalog and all official quality statistics are generated
by `2_singphotandmatching.ipynb`. Figures are saved and closed; notebook
output contains paths instead of inline plots.


## 1. Load Production Products


In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.table import MaskedColumn, Table
from astropy.wcs import FITSFixedWarning

warnings.filterwarnings("ignore", category=FITSFixedWarning)


def project_root():
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "Notebooks" else cwd


PROJECT_ROOT = project_root()
COSMOS = 1
BANDS_7DT = [f"m{i}" for i in range(400, 900, 25)]
DATA_ROOT = PROJECT_ROOT / "COSMOS_data"
MATCH_DIR = PROJECT_ROOT / f"output/COSMOS{COSMOS}/singlebandphot/matching"
FIG_DIR = MATCH_DIR / "figures"


def get_image_path(cosmos, band, weight=False):
    image_dir = DATA_ROOT / f"COSMOS{cosmos}"
    suffix = "weight.fits" if weight else "com.fits"
    paths = list(image_dir.glob(f"*{band}*.{suffix}"))
    if len(paths) != 1:
        raise FileNotFoundError(
            f"Expected one COSMOS{cosmos} {band} {suffix} file, found {len(paths)}"
        )
    return paths[0]


union_master = Table.read(MATCH_DIR / "master_7dt_finalcat_catalog.fits")
master = Table.read(MATCH_DIR / "master_catalog.fits")
full20 = Table.read(MATCH_DIR / "master_full20_catalog.fits")
band_uncertainty = Table.read(MATCH_DIR / "band_uncertainty.fits")
band_recovery = Table.read(MATCH_DIR / "finalcat_band_recovery.fits")
band_depths = Table.read(MATCH_DIR / "finalcat_band_depths.fits")
positional_quality = Table.read(MATCH_DIR / "full20_positional_quality.fits")
selection_counts = Table.read(MATCH_DIR / "positional_quality_selection_counts.fits")
sample_summary = Table.read(MATCH_DIR / "positional_quality_sample_summary.fits")
band_statistics = Table.read(MATCH_DIR / "positional_quality_band_stats.fits")
snr_coverage = Table.read(MATCH_DIR / "positional_quality_snr_coverage.fits")

master_class = np.asarray(master["master_class"]).astype(str)
matched = master[master_class == "matched"]
only_7dt = master[master_class == "7dt_only"]
only_finalcat = master[master_class == "finalcat_only"]
SAMPLE_NAMES = ["S1_star", "S2_all", "S3_galaxy", "S4_nonstar"]
SAMPLE_POPS = {
    name: positional_quality[
        np.asarray(positional_quality[f"sample_{name}"], dtype=bool)
    ]
    for name in SAMPLE_NAMES
}

print(f"master: {len(master)}")
print(f"full20: {len(full20)}")
print(f"positional-quality rows: {len(positional_quality)}")


master: 94388
full20: 1038
positional-quality rows: 20760


## 2. Temporary Visual Selections

The overlap box is an in-memory display selection. It is not written as a
catalog product.


In [2]:
RA_MIN, RA_MAX = 149.6, 150.3
DEC_MIN, DEC_MAX = 1.7, 2.3
ra = np.asarray(master["master_ra"], dtype=float)
dec = np.asarray(master["master_dec"], dtype=float)
overlap_mask = (
    (ra >= RA_MIN) & (ra <= RA_MAX)
    & (dec >= DEC_MIN) & (dec <= DEC_MAX)
)
master_overlap = master[overlap_mask]
overlap_class = np.asarray(master_overlap["master_class"]).astype(str)
matched_overlap = master_overlap[overlap_class == "matched"]
only_7dt_overlap = master_overlap[overlap_class == "7dt_only"]
only_finalcat_overlap = master_overlap[overlap_class == "finalcat_only"]
print(
    f"Overlap rows: {len(master_overlap)} "
    f"(matched={len(matched_overlap)}, 7DT-only={len(only_7dt_overlap)}, "
    f"FINALCAT-only={len(only_finalcat_overlap)})"
)


Overlap rows: 40859 (matched=2370, 7DT-only=8953, FINALCAT-only=29536)


## 3. Source Information Viewer

The viewer saves 20-band, detection-image, and color-diagnostic figures.
It reads the production FINALCAT and SDSS columns and performs no live query.


In [3]:
from IPython.display import display
from astropy.io import fits
from astropy.table import MaskedColumn
from astropy.visualization import ImageNormalize, ZScaleInterval
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

detection_image_path = DATA_ROOT / f"COSMOS{COSMOS}/COSMOS{COSMOS}_detectionimage.fits"
detection_image = fits.getdata(detection_image_path, memmap=True)
detection_norm = ImageNormalize(detection_image, interval=ZScaleInterval())
sample_header = fits.getheader(get_image_path(COSMOS, BANDS_7DT[0]))
PIXEL_SCALE_ARCSEC = float(proj_plane_pixel_scales(WCS(sample_header))[0] * 3600.0)
CLASS_COLORS = {"matched": "black", "7dt_only": "tab:red", "finalcat_only": "tab:blue"}
_BAND_IMAGE_CACHE = {}
_BAND_NORM_CACHE = None


def _text(value):
    if isinstance(value, (bytes, np.bytes_)):
        value = value.decode(errors="replace")
    return str(value).strip()


def _row_value(row, name):
    if name not in row.colnames or np.ma.is_masked(row[name]):
        return np.ma.masked
    return row[name]


def _row_float(row, name):
    value = _row_value(row, name)
    if np.ma.is_masked(value):
        return np.nan
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value if np.isfinite(value) else np.nan


def _display_value(row, name):
    value = _row_value(row, name)
    if np.ma.is_masked(value):
        return "--"
    if isinstance(value, (bytes, np.bytes_, str)):
        text = _text(value)
        return "--" if text in {"", "NN", "nan"} else text
    try:
        number = float(value)
    except (TypeError, ValueError):
        return _text(value)
    if not np.isfinite(number) or number in {-9.0, -99.0, -999.0}:
        return "--"
    return f"{number:.6g}"


def _band_images_and_norm(max_samples_per_band=5000):
    global _BAND_NORM_CACHE
    if not _BAND_IMAGE_CACHE:
        for band in BANDS_7DT:
            _BAND_IMAGE_CACHE[band] = fits.getdata(
                get_image_path(COSMOS, band), memmap=True
            )
    if _BAND_NORM_CACHE is None:
        samples = []
        for image in _BAND_IMAGE_CACHE.values():
            flat = image.reshape(-1)
            step = max(1, len(flat) // max_samples_per_band)
            sample = np.asarray(flat[::step][:max_samples_per_band], dtype=float)
            samples.append(sample[np.isfinite(sample)])
        pooled = np.concatenate(samples)
        if not len(pooled):
            raise ValueError("The 7DT images contain no finite pixels for ZScale.")
        vmin, vmax = ZScaleInterval(n_samples=len(pooled)).get_limits(pooled)
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin, vmax = np.percentile(pooled, [1, 99])
        if vmax <= vmin:
            raise ValueError("Cannot determine a non-degenerate shared 7DT image scale.")
        _BAND_NORM_CACHE = ImageNormalize(vmin=vmin, vmax=vmax)
    return _BAND_IMAGE_CACHE, _BAND_NORM_CACHE


def _crop_bounds(image, x, y, pixel_range):
    if pixel_range is None:
        return 0, image.shape[1], 0, image.shape[0]
    center_x = int(round(x - 1))
    center_y = int(round(y - 1))
    half = int(round(pixel_range))
    return (
        max(0, center_x - half), min(image.shape[1], center_x + half + 1),
        max(0, center_y - half), min(image.shape[0], center_y + half + 1),
    )


def _catalog_plot_data(cat, bounds):
    xmin, xmax, ymin, ymax = bounds
    xs = np.ma.filled(cat["master_x"], np.nan).astype(float)
    ys = np.ma.filled(cat["master_y"], np.nan).astype(float)
    classes = (
        np.char.strip(np.asarray(cat["master_class"]).astype(str))
        if "master_class" in cat.colnames
        else np.full(len(cat), "source", dtype="U16")
    )
    in_view = (
        np.isfinite(xs) & np.isfinite(ys)
        & (xs - 1 >= xmin) & (xs - 1 < xmax)
        & (ys - 1 >= ymin) & (ys - 1 < ymax)
    )
    return xs, ys, classes, in_view


def _add_catalog_overlay(ax, plot_data):
    xs, ys, classes, in_view = plot_data
    for source_class in np.unique(classes[in_view]):
        selected = in_view & (classes == source_class)
        ax.scatter(
            xs[selected], ys[selected], s=20, facecolors="none",
            edgecolors=CLASS_COLORS.get(source_class, "tab:red"), linewidths=0.7,
        )


def _photometry_array(cat, band, suffix="mag"):
    name = f"{band}_{suffix}"
    if name not in cat.colnames:
        return np.full(len(cat), np.nan)
    return np.ma.filled(cat[name], np.nan).astype(float)


def _plot_color_diagnostics(row, reference_catalog, cmd_bands, ccd_bands):
    if len(cmd_bands) != 2 or len(ccd_bands) != 3:
        raise ValueError("cmd_bands needs two bands and ccd_bands needs three bands.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    blue, red = cmd_bands
    blue_mag = _photometry_array(reference_catalog, blue)
    red_mag = _photometry_array(reference_catalog, red)
    cmd_valid = np.isfinite(blue_mag) & np.isfinite(red_mag)
    if cmd_valid.any():
        axes[0].scatter(
            blue_mag[cmd_valid] - red_mag[cmd_valid], red_mag[cmd_valid],
            s=5, color="black", alpha=0.3, linewidths=0, rasterized=True,
        )
        axes[0].invert_yaxis()
    else:
        axes[0].text(0.5, 0.5, "Insufficient reference photometry", ha="center", transform=axes[0].transAxes)
    selected_mags = [_row_float(row, f"{band}_mag") for band in cmd_bands]
    selected_errs = [_row_float(row, f"{band}_mag_err") for band in cmd_bands]
    if np.all(np.isfinite(selected_mags)):
        xerr = np.hypot(*selected_errs) if np.all(np.isfinite(selected_errs)) else None
        yerr = selected_errs[1] if np.isfinite(selected_errs[1]) else None
        axes[0].errorbar(
            selected_mags[0] - selected_mags[1], selected_mags[1],
            xerr=xerr, yerr=yerr, fmt="*", ms=14, color="tab:red",
            markeredgecolor="black", label="selected source", zorder=5,
        )
        axes[0].legend(loc="best")
    else:
        missing = [band for band, value in zip(cmd_bands, selected_mags) if not np.isfinite(value)]
        axes[0].text(0.03, 0.04, f"Selected source unavailable: {', '.join(missing)} non-detected", transform=axes[0].transAxes, color="tab:red")
    axes[0].set_xlabel(f"{blue} - {red}")
    axes[0].set_ylabel(red)
    axes[0].set_title("Color-Magnitude Diagram")

    blue, middle, red = ccd_bands
    blue_mag = _photometry_array(reference_catalog, blue)
    middle_mag = _photometry_array(reference_catalog, middle)
    red_mag = _photometry_array(reference_catalog, red)
    ccd_valid = np.isfinite(blue_mag) & np.isfinite(middle_mag) & np.isfinite(red_mag)
    if ccd_valid.any():
        axes[1].scatter(
            blue_mag[ccd_valid] - middle_mag[ccd_valid],
            middle_mag[ccd_valid] - red_mag[ccd_valid],
            s=5, color="black", alpha=0.3, linewidths=0, rasterized=True,
        )
    else:
        axes[1].text(0.5, 0.5, "Insufficient reference photometry", ha="center", transform=axes[1].transAxes)
    selected_mags = [_row_float(row, f"{band}_mag") for band in ccd_bands]
    selected_errs = [_row_float(row, f"{band}_mag_err") for band in ccd_bands]
    if np.all(np.isfinite(selected_mags)):
        xerr = np.hypot(selected_errs[0], selected_errs[1]) if np.all(np.isfinite(selected_errs[:2])) else None
        yerr = np.hypot(selected_errs[1], selected_errs[2]) if np.all(np.isfinite(selected_errs[1:])) else None
        axes[1].errorbar(
            selected_mags[0] - selected_mags[1], selected_mags[1] - selected_mags[2],
            xerr=xerr, yerr=yerr, fmt="*", ms=14, color="tab:red",
            markeredgecolor="black", label="selected source", zorder=5,
        )
        axes[1].legend(loc="best")
    else:
        missing = [band for band, value in zip(ccd_bands, selected_mags) if not np.isfinite(value)]
        axes[1].text(0.03, 0.04, f"Selected source unavailable: {', '.join(missing)} non-detected", transform=axes[1].transAxes, color="tab:red")
    axes[1].set_xlabel(f"{blue} - {middle}")
    axes[1].set_ylabel(f"{middle} - {red}")
    axes[1].set_title("Color-Color Diagram")
    fig.tight_layout()
    return fig, axes



def _external_tables(row):
    finalcat_fields = [
        ("finalcat_idx", "catalog row"),
        ("finalcat_ID", "ID"),
        ("finalcat_match_status", "match status"),
        ("finalcat_sep_arcsec", "separation (arcsec)"),
        ("finalcat_STELLARITY", "STELLARITY"),
        ("finalcat_Z_TOT_Z", "total redshift"),
        ("finalcat_Z_TOT_ZERR", "redshift error"),
        ("finalcat_Z_TOT_ZSOURCE", "redshift source"),
    ]
    if bool(_row_value(row, "has_finalcat")):
        finalcat_table = Table(
            rows=[(label, _display_value(row, name)) for name, label in finalcat_fields],
            names=["field", "value"],
        )
    else:
        finalcat_table = Table(
            rows=[("status", "No FINALCAT counterpart")], names=["field", "value"]
        )

    sdss_fields = [
        ("sdss_match_status", "match status"),
        ("sdss_sep_arcsec", "separation (arcsec)"),
        ("sdss_n_candidates", "master candidates"),
        ("sdss_n_master_candidates", "SDSS competitors"),
        ("sdss_objid", "objID"),
        ("sdss_type", "type"),
        ("sdss_clean", "clean"),
        ("sdss_mode", "mode"),
        ("sdss_psfMag_g", "PSF g"),
        ("sdss_psfMag_r", "PSF r"),
        ("sdss_psfMag_i", "PSF i"),
    ]
    if bool(_row_value(row, "has_sdss")):
        sdss_table = Table(
            rows=[(label, _display_value(row, name)) for name, label in sdss_fields],
            names=["field", "value"],
        )
    else:
        sdss_table = Table(
            rows=[("status", "No production SDSS counterpart")],
            names=["field", "value"],
        )
    return finalcat_table, sdss_table


def _save_figure(figure, description, filename, subfolder, dpi=160):
    output_dir = FIG_DIR / subfolder
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    figure.savefig(output_path, dpi=dpi, bbox_inches="tight")
    plt.close(figure)
    print(f"Saved {description} to {output_path}")
    return output_path


def show_source_info_from_catalog(
    cat, source_id=None, pixel_range=40, cmd_bands=("m475", "m625"),
    ccd_bands=("m475", "m625", "m775"), reference_catalog=None,
):
    if len(cat) == 0:
        raise ValueError("The input catalog is empty.")
    required = {"master_id", "master_x", "master_y"}
    missing = required.difference(cat.colnames)
    if missing:
        raise KeyError(f"Catalog is missing required columns: {sorted(missing)}")
    if pixel_range is not None and (not np.isfinite(pixel_range) or pixel_range <= 0):
        raise ValueError("pixel_range must be None or a positive number.")

    catalog_ids = np.asarray(cat["master_id"], dtype=np.int64)
    if source_id is None:
        selected_index = int(np.random.default_rng().integers(len(cat)))
    else:
        matches = np.flatnonzero(catalog_ids == int(source_id))
        if not len(matches):
            raise KeyError(f"master_id={source_id} is not present in the catalog.")
        selected_index = int(matches[0])
    master_id = int(catalog_ids[selected_index])
    authoritative_matches = np.flatnonzero(
        np.asarray(master["master_id"], dtype=np.int64) == master_id
    )
    if not len(authoritative_matches):
        raise KeyError(f"master_id={master_id} is not present in the master catalog.")
    row = master[int(authoritative_matches[0])]
    selected_x = _row_float(row, "master_x")
    selected_y = _row_float(row, "master_y")
    if not np.isfinite(selected_x) or not np.isfinite(selected_y):
        raise ValueError("The selected source has a non-finite pixel position.")
    if reference_catalog is None:
        reference_catalog = master_overlap

    band_images, band_norm = _band_images_and_norm()
    first_bounds = _crop_bounds(
        band_images[BANDS_7DT[0]], selected_x, selected_y, pixel_range
    )
    plot_data = _catalog_plot_data(cat, first_bounds)
    band_figure, band_axes = plt.subplots(4, 5, figsize=(15, 12), sharex=True, sharey=True)
    for band, ax in zip(BANDS_7DT, band_axes.flat):
        image = band_images[band]
        bounds = _crop_bounds(image, selected_x, selected_y, pixel_range)
        xmin, xmax, ymin, ymax = bounds
        ax.imshow(
            image[ymin:ymax, xmin:xmax], origin="lower", cmap="gray", norm=band_norm,
            extent=[xmin + 1, xmax + 1, ymin + 1, ymax + 1],
        )
        overlay = plot_data if bounds == first_bounds else _catalog_plot_data(cat, bounds)
        _add_catalog_overlay(ax, overlay)
        detected = not np.ma.is_masked(_row_value(row, f"{band}_source_id"))
        ax.scatter(
            selected_x, selected_y, marker="+", s=100,
            color="cyan" if detected else "gray", linewidths=1.3,
        )
        ax.set_title(f"{band} | {'detected' if detected else 'non-detected'}", fontsize=10)
        ax.tick_params(labelsize=7)
    for ax in band_axes[-1, :]:
        ax.set_xlabel("X pixel")
    for ax in band_axes[:, 0]:
        ax.set_ylabel("Y pixel")
    band_figure.suptitle(f"7DT 20-band cutouts: master_id={master_id}", fontsize=14)
    band_figure.tight_layout(rect=[0, 0, 1, 0.97])
    band_path = _save_figure(
        band_figure, f"20-band cutouts for master_id={master_id}",
        f"source_{master_id}_7dt_cutouts.png",
        "source_viewer",
    )

    detection_bounds = _crop_bounds(detection_image, selected_x, selected_y, pixel_range)
    xmin, xmax, ymin, ymax = detection_bounds
    detection_figure, detection_ax = plt.subplots(figsize=(7, 7))
    detection_ax.imshow(
        detection_image[ymin:ymax, xmin:xmax], origin="lower", cmap="gray",
        norm=detection_norm, extent=[xmin + 1, xmax + 1, ymin + 1, ymax + 1],
    )
    _add_catalog_overlay(detection_ax, _catalog_plot_data(cat, detection_bounds))
    detection_ax.scatter(
        selected_x, selected_y, marker="+", s=180, color="cyan", linewidths=1.5,
        label=f"master_id={master_id}",
    )
    source_class = _text(_row_value(row, "master_class"))
    detection_ax.set_title(f"Detection image: {source_class}, master_id={master_id}")
    detection_ax.set_xlabel("X pixel")
    detection_ax.set_ylabel("Y pixel")
    detection_ax.legend(loc="upper right")
    detection_figure.tight_layout()
    detection_path = _save_figure(
        detection_figure, f"detection-image cutout for master_id={master_id}",
        f"source_{master_id}_detection.png",
        "source_viewer",
    )

    color_figure, color_axes = _plot_color_diagnostics(
        row, reference_catalog, cmd_bands, ccd_bands
    )
    color_path = _save_figure(
        color_figure, f"color diagnostics for master_id={master_id}",
        f"source_{master_id}_color_diagnostics.png",
        "source_viewer",
    )

    band_fwhm = {
        _text(band): float(value)
        for band, value in zip(band_uncertainty["band"], band_uncertainty["fwhm_pix"])
    }
    band_rows = []
    extendicities = []
    for band in BANDS_7DT:
        source_value = _row_value(row, f"{band}_source_id")
        detected = not np.ma.is_masked(source_value)
        radius = _row_float(row, f"{band}_flux_radius") if detected else np.nan
        source_fwhm = 2.0 * radius if np.isfinite(radius) and radius > 0 else np.nan
        reference_fwhm = band_fwhm.get(band, np.nan)
        extendicity = (
            source_fwhm / reference_fwhm
            if np.isfinite(source_fwhm) and reference_fwhm > 0 else np.nan
        )
        if np.isfinite(extendicity):
            extendicities.append(extendicity)
        band_rows.append((
            band, detected, int(source_value) if detected else np.ma.masked,
            _row_value(row, f"{band}_mag") if detected else np.ma.masked,
            _row_value(row, f"{band}_mag_err") if detected else np.ma.masked,
            _row_value(row, f"{band}_ellipticity") if detected else np.ma.masked,
            radius if np.isfinite(radius) else np.ma.masked,
            source_fwhm if np.isfinite(source_fwhm) else np.ma.masked,
            source_fwhm * PIXEL_SCALE_ARCSEC if np.isfinite(source_fwhm) else np.ma.masked,
            reference_fwhm if np.isfinite(reference_fwhm) else np.ma.masked,
            reference_fwhm * PIXEL_SCALE_ARCSEC if np.isfinite(reference_fwhm) else np.ma.masked,
            extendicity if np.isfinite(extendicity) else np.ma.masked,
        ))
    band_table = Table(
        rows=band_rows, masked=True,
        names=["band", "detected", "source_id", "mag", "mag_err", "ellipticity",
               "flux_radius_pix", "source_fwhm_pix", "source_fwhm_arcsec",
               "band_fwhm_pix", "band_fwhm_arcsec", "extendicity"],
    )
    for name in (
        "mag", "mag_err", "ellipticity", "flux_radius_pix", "source_fwhm_pix",
        "source_fwhm_arcsec", "band_fwhm_pix", "band_fwhm_arcsec", "extendicity",
    ):
        band_table[name].format = ".4f"
    median_extendicity = float(np.median(extendicities)) if extendicities else np.nan
    mad_extendicity = (
        float(np.median(np.abs(np.asarray(extendicities) - median_extendicity)))
        if extendicities else np.nan
    )
    summary_table = Table(masked=True)
    summary_table["master_id"] = [master_id]
    summary_table["master_class"] = [source_class]
    summary_table["n_7dt_bands"] = [int(_row_value(row, "n_7dt_bands"))]
    summary_table["master_x"] = [selected_x]
    summary_table["master_y"] = [selected_y]
    summary_table["master_ra"] = [_row_float(row, "master_ra")]
    summary_table["master_dec"] = [_row_float(row, "master_dec")]
    summary_table["n_extendicity_bands"] = [len(extendicities)]
    summary_table["extendicity_median"] = MaskedColumn(
        [median_extendicity], mask=[not extendicities]
    )
    summary_table["extendicity_mad"] = MaskedColumn(
        [mad_extendicity], mask=[not extendicities]
    )
    finalcat_table, sdss_table = _external_tables(row)

    print("Source summary")
    display(summary_table)
    print("7DT band measurements")
    display(band_table)
    print("FINALCAT information")
    display(finalcat_table)
    print("SDSS information")
    display(sdss_table)
    return {
        "master_id": master_id,
        "band_path": band_path,
        "detection_path": detection_path,
        "color_path": color_path,
        "summary_table": summary_table,
        "band_table": band_table,
        "finalcat_table": finalcat_table,
        "sdss_table": sdss_table,
    }


In [4]:
source_info = show_source_info_from_catalog(
    master, source_id=1317, pixel_range=40
)


Saved 20-band cutouts for master_id=1317 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/source_viewer/source_1317_7dt_cutouts.png
Saved detection-image cutout for master_id=1317 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/source_viewer/source_1317_detection.png
Saved color diagnostics for master_id=1317 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/source_viewer/source_1317_color_diagnostics.png
Source summary


master_id,master_class,n_7dt_bands,master_x,master_y,master_ra,master_dec,n_extendicity_bands,extendicity_median,extendicity_mad
int64,str8,int64,float64,float64,float64,float64,int64,float64,float64
1317,7dt_only,1,1271.1146240234375,2442.86572265625,150.33743026511425,1.815585617959336,1,0.22074157309831502,0.0


7DT band measurements


band,detected,source_id,mag,mag_err,ellipticity,flux_radius_pix,source_fwhm_pix,source_fwhm_arcsec,band_fwhm_pix,band_fwhm_arcsec,extendicity
str4,bool,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64
m400,True,6221,22.7832,0.8764,0.5773,0.5858,1.1716,0.5916,5.3074,2.6802,0.2207
m425,False,--,--,--,--,--,--,--,5.8914,2.9751,--
m450,False,--,--,--,--,--,--,--,5.7651,2.9114,--
m475,False,--,--,--,--,--,--,--,6.1157,3.0884,--
m500,False,--,--,--,--,--,--,--,5.0001,2.5251,--
m525,False,--,--,--,--,--,--,--,5.5404,2.7979,--
m550,False,--,--,--,--,--,--,--,6.2455,3.1540,--
m575,False,--,--,--,--,--,--,--,6.6446,3.3555,--
m600,False,--,--,--,--,--,--,--,5.7702,2.9140,--


FINALCAT information


field,value
str6,str23
status,No FINALCAT counterpart


SDSS information


field,value
str6,str30
status,No production SDSS counterpart


## 4. Nearest-source Search


In [5]:
# SearchNearSources(mastercat, pos) usage:
#
# result = SearchNearSources(mastercat, pos)
#     mastercat: master catalog or any subset (e.g. master, matched_overlap,
#         only_7dt_overlap, only_finalcat_overlap). Must contain master_id,
#         master_x, master_y in the FITS 1-based pixel convention.
#     pos: single query position as [x, y] or (x, y), in the same
#         master_x/master_y coordinate system. No origin conversion is
#         applied because query and catalog share the coordinate frame.
#
# Returns an astropy Table with one row per neighbor (up to 10):
#     master_id, x, y, pixel_distance, sorted by ascending distance.
# Also prints the query, the catalog size, and the result table.
# Example: SearchNearSources(only_7dt_overlap, [3325, 4880])

def SearchNearSources(mastercat, pos):
    required = {"master_id", "master_x", "master_y"}
    missing = required.difference(mastercat.colnames)
    if missing:
        raise KeyError(f"Catalog is missing required columns: {sorted(missing)}")
    if len(mastercat) == 0:
        raise ValueError("The input catalog is empty.")
    query = np.asarray(pos, dtype=float)
    if query.shape != (2,) or not np.all(np.isfinite(query)):
        raise ValueError("pos must be a single finite [x, y] pair.")
    query_x, query_y = float(query[0]), float(query[1])
    master_ids = np.asarray(mastercat["master_id"]).astype(int)
    xs = np.ma.filled(mastercat["master_x"], np.nan).astype(float)
    ys = np.ma.filled(mastercat["master_y"], np.nan).astype(float)
    finite = np.isfinite(xs) & np.isfinite(ys)
    if not np.any(finite):
        raise ValueError("No catalog rows with finite master_x/master_y.")
    valid_idx = np.flatnonzero(finite)
    distances = np.hypot(xs[finite] - query_x, ys[finite] - query_y)
    order = np.argsort(distances, kind="stable")[:10]
    chosen = valid_idx[order]
    result = Table()
    result["master_id"] = master_ids[chosen]
    result["x"] = xs[chosen]
    result["y"] = ys[chosen]
    result["pixel_distance"] = distances[order]
    print(f"Catalog sources (finite position): {len(valid_idx)}/{len(mastercat)}")
    print(f"Query: x={query_x:.2f}, y={query_y:.2f}")
    print(f"Returning {len(result)} nearest sources")
    result.pprint(max_width=-1)
    return result


In [6]:
nearest_sources = SearchNearSources(only_7dt_overlap, (5755, 4445))


Catalog sources (finite position): 8953/8953
Query: x=5755.00, y=4445.00
Returning 10 nearest sources
master_id         x                 y           pixel_distance  
--------- ----------------- ----------------- ------------------
    13074 5757.070831420936 4443.290796941828  2.685091780181627
    33470 5757.500976562499 4440.734374999999  4.944738659100452
     2446 5740.958225013362 4429.521516055875  20.89868201547478
    10443   5735.1767578125  4455.04052734375   22.2210062815542
    32044 5760.853515624999 4419.333984374999 26.325045170614604
    13073       5724.203125 4436.419433593751 31.969886293474293
    13071       5722.265625  4432.10107421875 35.184109949675026
    13065 5729.671875000001      4417.8671875  37.11742758025851
    17793   5752.1630859375   4483.2568359375  38.36187661398751
    15974  5717.97216796875  4430.86572265625  39.63380048600573


## 5. Single-source Positional Diagnostics

Residuals and tolerances are loaded from the production positional-quality
table. This section only renders them.


In [7]:
def plot_point_source_positions(master_id, sample_label=None):
    master_id = int(master_id)
    choose = np.asarray(positional_quality["master_id"], dtype=np.int64) == master_id
    table = positional_quality[choose]
    if len(table) != len(BANDS_7DT):
        raise KeyError(f"Expected 20 positional rows for master_id={master_id}")
    source_rows = np.flatnonzero(
        np.asarray(full20["master_id"], dtype=np.int64) == master_id
    )
    if len(source_rows) != 1:
        raise KeyError(f"master_id={master_id} is not in the full20 catalog")
    source = full20[int(source_rows[0])]
    order = np.argsort(np.asarray(table["wavelength_nm"], dtype=int))
    table = table[order]
    delta_ra = np.asarray(table["delta_ra_arcsec"], dtype=float)
    delta_dec = np.asarray(table["delta_dec_arcsec"], dtype=float)
    wavelength = np.asarray(table["wavelength_nm"], dtype=float)

    figure = plt.figure(figsize=(10, 9))
    grid = figure.add_gridspec(
        2, 2, width_ratios=(4, 1.2), height_ratios=(4, 1.2),
        hspace=0.05, wspace=0.05,
    )
    main_ax = figure.add_subplot(grid[0, 0])
    right_ax = figure.add_subplot(grid[0, 1], sharey=main_ax)
    bottom_ax = figure.add_subplot(grid[1, 0], sharex=main_ax)
    scatter = main_ax.scatter(
        delta_ra, delta_dec, c=wavelength, cmap="viridis",
        s=65, edgecolor="black", linewidth=0.5, zorder=3,
    )
    radial_offset = np.hypot(delta_ra, delta_dec)
    for index in np.argsort(radial_offset)[-3:]:
        main_ax.annotate(
            str(table["band"][index]), (delta_ra[index], delta_dec[index]),
            xytext=(4, 4), textcoords="offset points", fontsize=8,
        )
    main_ax.scatter(
        0, 0, marker="+", s=180, linewidths=1.5, color="black",
        label="7DT weighted master", zorder=4,
    )
    reference_radius = float(
        np.nanmedian(3.0 * np.asarray(table["loo_predicted_sigma_arcsec"], dtype=float))
    )
    main_ax.axhline(0, color="0.75", linewidth=0.7)
    main_ax.axvline(0, color="0.75", linewidth=0.7)
    main_ax.text(
        0.02, 0.02, f"median LOO 3-sigma = {reference_radius:.3f} arcsec",
        transform=main_ax.transAxes, color="tab:red", fontsize=9,
    )
    limit = max(0.05, np.max(np.abs(np.r_[delta_ra, delta_dec])) * 1.35)
    main_ax.set_xlim(-limit, limit)
    main_ax.set_ylim(-limit, limit)
    main_ax.set_aspect("equal", adjustable="box")
    main_ax.set_ylabel("Delta Dec [arcsec]")
    main_ax.legend(loc="upper right", fontsize=8)
    bottom_ax.scatter(
        delta_ra, np.zeros(len(delta_ra)), c=wavelength, cmap="viridis",
        s=55, edgecolor="black", linewidth=0.4,
    )
    bottom_ax.axvline(0, color="0.75", linewidth=0.7)
    bottom_ax.set_ylim(-1, 1)
    bottom_ax.set_yticks([])
    bottom_ax.set_xlabel("Delta RA cos(Dec) [arcsec]")
    bottom_ax.text(0.02, 0.82, "1D Delta RA", transform=bottom_ax.transAxes, fontsize=9)
    right_ax.scatter(
        np.zeros(len(delta_dec)), delta_dec, c=wavelength, cmap="viridis",
        s=55, edgecolor="black", linewidth=0.4,
    )
    right_ax.axhline(0, color="0.75", linewidth=0.7)
    right_ax.set_xlim(-1, 1)
    right_ax.set_xticks([])
    right_ax.tick_params(labelleft=False)
    right_ax.text(
        0.08, 0.97, "1D Delta Dec", transform=right_ax.transAxes,
        va="top", fontsize=9,
    )
    colorbar = figure.colorbar(scatter, ax=[main_ax, right_ax, bottom_ax], pad=0.02)
    colorbar.set_label("Wavelength [nm]")
    label = f"{sample_label}, " if sample_label else ""
    figure.suptitle(
        f"{label}7DT band offsets: master_id={master_id}, "
        f"SDSS separation={_display_value(source, 'sdss_sep_arcsec')} arcsec"
    )
    stem = f"point_source_offsets_master_{master_id}"
    if sample_label:
        stem = f"{sample_label}_offsets_master_{master_id}"
    path = _save_figure(
        figure, f"positional offsets for {label}master_id={master_id}",
        stem + ".png",
        "positional_single",
    )
    return {"source": source, "table": table, "path": path}


In [8]:
star_rows = full20[np.asarray(full20["sample_S1_star"], dtype=bool)]
example_index = int(np.argmin(np.asarray(star_rows["sdss_sep_arcsec"], dtype=float)))
POINT_SOURCE_MASTER_ID = int(star_rows["master_id"][example_index])
point_source_result = plot_point_source_positions(POINT_SOURCE_MASTER_ID)


Saved positional offsets for master_id=1950 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/point_source_offsets_master_1950.png


## 6. Population Visualizations

Official statistics are loaded from production. Plot-specific binning and
rendering remain here.


In [9]:
star_population = SAMPLE_POPS["S1_star"]
band_values = [
    np.asarray(
        star_population["loo_normalized_residual"][
            np.asarray(star_population["band"]).astype(str) == band
        ], dtype=float,
    )
    for band in BANDS_7DT
]
all_normalized = np.asarray(star_population["loo_normalized_residual"], dtype=float)
rayleigh_grid = np.linspace(0, max(4.5, np.percentile(all_normalized, 99)), 500)
rayleigh_density = rayleigh_grid * np.exp(-0.5 * rayleigh_grid**2)
figure, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].boxplot(band_values, positions=np.arange(len(BANDS_7DT)), showfliers=False)
axes[0].axhline(np.sqrt(2.0 * np.log(2.0)), color="tab:blue", linestyle="--", label="Rayleigh median")
axes[0].axhline(3.0, color="tab:red", linestyle=":", label="D = 3")
axes[0].set_xticks(np.arange(len(BANDS_7DT)), BANDS_7DT, rotation=45)
axes[0].set(xlabel="Band", ylabel="LOO normalized radial residual D", title="Normalized residual by band")
axes[0].legend()
axes[1].hist(all_normalized, bins=60, density=True, histtype="step", color="black", label="SDSS point sources")
axes[1].plot(rayleigh_grid, rayleigh_density, color="tab:blue", label="Unit Rayleigh prediction")
axes[1].axvline(3.0, color="tab:red", linestyle=":")
axes[1].set(xlabel="LOO normalized radial residual D", ylabel="Density", title="All bands and point sources")
axes[1].legend()
figure.tight_layout()
_ = _save_figure(
    figure, "SDSS point-source normalized residuals",
    "sdss_point_source_normalized_residuals.png",
    "positional_population",
)


Saved SDSS point-source normalized residuals to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_population/sdss_point_source_normalized_residuals.png


## 7. FINALCAT-referenced Recovery

These figures directly render the production recovery and depth tables. No
catalog matching, binning, fitting, or depth estimation is performed here.

In [10]:
figure, axes = plt.subplots(4, 5, figsize=(18, 13), sharex=True, sharey=True)
for axis, band in zip(axes.flat, BANDS_7DT):
    rows = band_recovery[np.char.strip(np.asarray(band_recovery["band"]).astype(str)) == band]
    magnitude = np.asarray(rows["mag_center"], dtype=float)
    raw = np.asarray(rows["recovery_fraction"], dtype=float)
    error = np.asarray(rows["recovery_error"], dtype=float)
    monotonic = np.asarray(rows["recovery_monotonic"], dtype=float)
    used = np.asarray(rows["used_for_depth"], dtype=bool)
    axis.errorbar(magnitude, raw, yerr=error, fmt=".", color="0.65", alpha=0.8, label="raw")
    axis.plot(magnitude[used], monotonic[used], color="black", linewidth=1.5, label="monotonic")
    depth_rows = band_depths[
        np.char.strip(np.asarray(band_depths["band"]).astype(str)) == band
    ]
    for depth_row, color in zip(depth_rows, ("tab:blue", "tab:orange", "tab:red")):
        if np.ma.is_masked(depth_row["depth_mag"]):
            continue
        depth = float(depth_row["depth_mag"])
        if np.isfinite(depth):
            axis.axvline(depth, color=color, linewidth=0.9, alpha=0.8)
    reference_filter = str(rows["reference_filter"][0]).strip()
    axis.set_title(f"{band} | FINALCAT {reference_filter}")
    axis.set_xlim(18, 29)
    axis.set_ylim(-0.03, 1.03)
    axis.grid(alpha=0.15)
for axis in axes[-1, :]:
    axis.set_xlabel("FINALCAT reference magnitude")
for axis in axes[:, 0]:
    axis.set_ylabel("Recovery fraction")
handles, labels = axes.flat[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=2)
figure.suptitle("Per-band 7DT recovery of in-footprint FINALCAT sources", y=0.995)
figure.tight_layout(rect=(0, 0, 1, 0.975))
_ = _save_figure(figure, "FINALCAT-referenced band recovery", "finalcat_band_recovery.png", "finalcat_recovery")

figure, axis = plt.subplots(figsize=(11, 6))
threshold_colors = {0.5: "tab:blue", 0.8: "tab:orange", 0.9: "tab:red"}
for threshold, color in threshold_colors.items():
    rows = band_depths[np.isclose(np.asarray(band_depths["recovery_threshold"], dtype=float), threshold)]
    order = np.argsort(np.asarray(rows["wavelength_nm"], dtype=float))
    rows = rows[order]
    axis.plot(
        rows["wavelength_nm"], rows["depth_mag"], marker="o", color=color,
        label=f"{int(100 * threshold)}% recovery",
    )
axis.set(
    xlabel="7DT central wavelength [nm]",
    ylabel="FINALCAT reference magnitude at crossing",
    title="FINALCAT-referenced 7DT recovery depths",
)
axis.grid(alpha=0.2)
axis.legend()
figure.tight_layout()
_ = _save_figure(figure, "FINALCAT-referenced completeness depths", "finalcat_band_depths.png", "finalcat_recovery")


Saved FINALCAT-referenced band recovery to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/finalcat_recovery/finalcat_band_recovery.png
Saved FINALCAT-referenced completeness depths to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/finalcat_recovery/finalcat_band_depths.png


## 8. Random Representative Sources


In [11]:
representative_rng = np.random.default_rng()
representative_ids = {}
for sample_name in SAMPLE_NAMES:
    rows = full20[np.asarray(full20[f"sample_{sample_name}"], dtype=bool)]
    n_draw = min(3, len(rows))
    representative_ids[sample_name] = [
        int(value) for value in representative_rng.choice(
            np.asarray(rows["master_id"], dtype=np.int64),
            size=n_draw, replace=False,
        )
    ]
print(representative_ids)
for sample_name, master_ids in representative_ids.items():
    for master_id in master_ids:
        plot_point_source_positions(master_id, sample_label=sample_name)


{'S1_star': [2740, 2417, 845], 'S2_all': [3063, 3163, 907], 'S3_galaxy': [2838, 2614, 2074], 'S4_nonstar': [2415, 2902, 1082]}
Saved positional offsets for S1_star, master_id=2740 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S1_star_offsets_master_2740.png


Saved positional offsets for S1_star, master_id=2417 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S1_star_offsets_master_2417.png


Saved positional offsets for S1_star, master_id=845 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S1_star_offsets_master_845.png
Saved positional offsets for S2_all, master_id=3063 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S2_all_offsets_master_3063.png


Saved positional offsets for S2_all, master_id=3163 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S2_all_offsets_master_3163.png


Saved positional offsets for S2_all, master_id=907 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S2_all_offsets_master_907.png
Saved positional offsets for S3_galaxy, master_id=2838 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S3_galaxy_offsets_master_2838.png


Saved positional offsets for S3_galaxy, master_id=2614 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S3_galaxy_offsets_master_2614.png


Saved positional offsets for S3_galaxy, master_id=2074 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S3_galaxy_offsets_master_2074.png
Saved positional offsets for S4_nonstar, master_id=2415 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S4_nonstar_offsets_master_2415.png


Saved positional offsets for S4_nonstar, master_id=2902 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S4_nonstar_offsets_master_2902.png


Saved positional offsets for S4_nonstar, master_id=1082 to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/positional_single/S4_nonstar_offsets_master_1082.png


## 9. Sample-comparison Figures


In [12]:
colors = dict(zip(SAMPLE_NAMES, ["tab:blue", "black", "tab:orange", "tab:green"]))

figure, axes = plt.subplots(1, 2, figsize=(16, 5))
pooled = [
    np.asarray(SAMPLE_POPS[name]["loo_normalized_residual"], dtype=float)
    for name in SAMPLE_NAMES
]
finite_all = np.concatenate(pooled)
bins = np.linspace(0, max(4.5, np.percentile(finite_all, 99.5)), 70)
for name, values in zip(SAMPLE_NAMES, pooled):
    axes[0].hist(values, bins=bins, density=True, histtype="step", color=colors[name], label=name)
grid = np.linspace(0, bins[-1], 500)
axes[0].plot(grid, grid * np.exp(-0.5 * grid**2), color="tab:red", linestyle="--", label="Unit Rayleigh")
axes[0].set(xlabel="LOO normalized residual D", ylabel="Density", title="Sample distributions")
axes[0].legend()
for name in SAMPLE_NAMES:
    rows = band_statistics[np.asarray(band_statistics["sample"]).astype(str) == name]
    axes[1].plot(rows["wavelength_nm"], rows["median_D"], marker="o", color=colors[name], label=name)
axes[1].axhline(np.sqrt(2.0 * np.log(2.0)), color="tab:red", linestyle="--", label="Rayleigh median")
axes[1].set(xlabel="Wavelength [nm]", ylabel="Median D", title="Median normalized residual by band")
axes[1].legend()
figure.tight_layout()
_ = _save_figure(figure, "four-sample normalized-residual comparison", "four_sample_normalized_residuals.png", "sample_comparison")

figure, axes = plt.subplots(1, 2, figsize=(16, 5))
for name in SAMPLE_NAMES:
    rows = band_statistics[np.asarray(band_statistics["sample"]).astype(str) == name]
    axes[0].plot(rows["wavelength_nm"], rows["loo_ra_nmad"], marker="o", color=colors[name], label=name)
    axes[1].plot(rows["wavelength_nm"], rows["loo_dec_nmad"], marker="o", color=colors[name], label=name)
axes[0].set(xlabel="Wavelength [nm]", ylabel="NMAD [arcsec]", title="LOO RA scatter")
axes[1].set(xlabel="Wavelength [nm]", ylabel="NMAD [arcsec]", title="LOO Dec scatter")
for ax in axes:
    ax.legend()
figure.tight_layout()
_ = _save_figure(figure, "four-sample band-wise NMAD comparison", "four_sample_band_nmad.png", "sample_comparison")

figure, axes = plt.subplots(1, 2, figsize=(16, 5))
for name in SAMPLE_NAMES:
    pop = SAMPLE_POPS[name]
    axes[0].hist(np.asarray(pop["loo_delta_ra_arcsec"], dtype=float), bins=70, density=True, histtype="step", color=colors[name], label=name)
    axes[1].hist(np.asarray(pop["loo_delta_dec_arcsec"], dtype=float), bins=70, density=True, histtype="step", color=colors[name], label=name)
axes[0].set(xlabel="LOO Delta RA cos(Dec) [arcsec]", ylabel="Density", title="Pooled RA offsets")
axes[1].set(xlabel="LOO Delta Dec [arcsec]", ylabel="Density", title="Pooled Dec offsets")
for ax in axes:
    ax.legend()
figure.tight_layout()
_ = _save_figure(figure, "four-sample pooled offset distributions", "four_sample_offset_histograms.png", "sample_comparison")

figure, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, name in zip(axes.flat, SAMPLE_NAMES):
    pop = SAMPLE_POPS[name]
    snr = np.asarray(pop["snrratio"], dtype=float)
    residual = np.asarray(pop["loo_residual_arcsec"], dtype=float)
    finite = np.isfinite(snr) & np.isfinite(residual) & (snr > 0) & (residual > 0)
    ax.hexbin(snr[finite], residual[finite], gridsize=45, bins="log", mincnt=1, cmap="viridis")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_xlabel("SourceXtractor++ S/N")
    ax.set_ylabel("LOO radial residual [arcsec]")
figure.tight_layout()
_ = _save_figure(figure, "S/N versus LOO radial residual", "four_sample_snr_vs_residual.png", "sample_comparison")


Saved four-sample normalized-residual comparison to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/sample_comparison/four_sample_normalized_residuals.png


Saved four-sample band-wise NMAD comparison to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/sample_comparison/four_sample_band_nmad.png
Saved four-sample pooled offset distributions to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/sample_comparison/four_sample_offset_histograms.png


Saved S/N versus LOO radial residual to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/sample_comparison/four_sample_snr_vs_residual.png


## 10. Detection-image Catalog Overlays

The original `plot_ImageAndDetectedSources` method from
`1_multibandphot.ipynb` is adapted here for the production master catalog.
The full detection image is divided into a 4 x 5 grid. Each tile is saved
with `7dt_only` sources in red and `matched` sources in black.


In [13]:
from astropy.visualization import ImageNormalize, ZScaleInterval

detection_image_path = DATA_ROOT / f"COSMOS{COSMOS}/COSMOS{COSMOS}_detectionimage.fits"
detection_image = fits.getdata(detection_image_path, memmap=True)

def plot_ImageAndDetectedSources(
    image, cat, center_point=None, length=None,
    x_column="pixel_centroid_x", y_column="pixel_centroid_y",
    source_color="red", title=None, output_filename=None,
):
    """Plot a detection-image cutout and catalog source positions."""
    required = {x_column, y_column}
    missing = required.difference(cat.colnames)
    if missing:
        raise KeyError(f"Catalog is missing required columns: {sorted(missing)}")
    if (center_point is None) != (length is None):
        raise ValueError("center_point and length must be provided together.")

    if center_point is None:
        xmin, ymin = 0, 0
        xmax, ymax = image.shape[1], image.shape[0]
    else:
        half = float(length) / 2.0
        xmin = max(0, int(np.floor(center_point[0] - half)))
        xmax = min(image.shape[1], int(np.ceil(center_point[0] + half)))
        ymin = max(0, int(np.floor(center_point[1] - half)))
        ymax = min(image.shape[0], int(np.ceil(center_point[1] + half)))
    if xmax <= xmin or ymax <= ymin:
        raise ValueError("The requested cutout is empty.")

    x = np.ma.filled(cat[x_column], np.nan).astype(float)
    y = np.ma.filled(cat[y_column], np.nan).astype(float)
    finite = (
        np.isfinite(x) & np.isfinite(y)
        & (x >= xmin) & (x < xmax) & (y >= ymin) & (y < ymax)
    )
    image_plot = image[ymin:ymax, xmin:xmax]
    figure, axis = plt.subplots(figsize=(10, 10))
    axis.imshow(
        image_plot, origin="lower", cmap="gray",
        norm=ImageNormalize(image_plot, interval=ZScaleInterval()),
        extent=[xmin, xmax, ymin, ymax],
    )
    axis.scatter(
        x[finite], y[finite], s=36, facecolors="none",
        edgecolors=source_color, linewidths=0.8,
        label=f"detected sources ({int(finite.sum())})", rasterized=True,
    )
    if center_point is not None:
        axis.axhline(center_point[1], color="cyan", linestyle="--", linewidth=0.8)
        axis.axvline(center_point[0], color="cyan", linestyle="--", linewidth=0.8)
    axis.set(xlabel="X pixel", ylabel="Y pixel")
    if title is not None:
        axis.set_title(title)
    axis.legend(loc="upper right")
    figure.tight_layout()
    if output_filename is not None:
        output_path = _save_figure(figure, title or "detection-image catalog overlay", output_filename, "detection_tiles")
    else:
        output_path = None
    return figure, axis, output_path, int(finite.sum())

master_classes = np.char.strip(np.asarray(master["master_class"]).astype(str))
plot_catalog = master[
    np.isin(master_classes, ["7dt_only", "matched"])
]
plot_catalog["plot_label"] = np.where(
    np.asarray(plot_catalog["master_class"]).astype(str) == "7dt_only",
    "7DT-only", "matched",
)

n_rows, n_columns = 4, 5
y_edges = np.linspace(0, detection_image.shape[0], n_rows + 1, dtype=int)
x_edges = np.linspace(0, detection_image.shape[1], n_columns + 1, dtype=int)
for tile_index, (row_index, column_index) in enumerate(
    ((r, c) for r in range(n_rows) for c in range(n_columns)), start=1
):
    xmin, xmax = x_edges[column_index], x_edges[column_index + 1]
    ymin, ymax = y_edges[row_index], y_edges[row_index + 1]
    center = ((xmin + xmax) / 2.0, (ymin + ymax) / 2.0)
    tile_length = max(xmax - xmin, ymax - ymin)
    tile = plot_catalog.copy(copy_data=True)
    colors = np.where(
        np.asarray(tile["plot_label"]).astype(str) == "7DT-only",
        "tab:red", "black",
    )
    # Plot the two classes separately so the legend identifies both groups.
    figure, axis = plt.subplots(figsize=(10, 10))
    axis.imshow(
        detection_image[ymin:ymax, xmin:xmax], origin="lower", cmap="gray",
        norm=ImageNormalize(detection_image[ymin:ymax, xmin:xmax], interval=ZScaleInterval()),
        extent=[xmin, xmax, ymin, ymax],
    )
    for label, color in (("7DT-only", "tab:red"), ("matched", "black")):
        choose = np.asarray(tile["plot_label"]).astype(str) == label
        x = np.ma.filled(tile["master_x"], np.nan).astype(float)
        y = np.ma.filled(tile["master_y"], np.nan).astype(float)
        choose &= np.isfinite(x) & np.isfinite(y)
        choose &= (x >= xmin) & (x < xmax) & (y >= ymin) & (y < ymax)
        axis.scatter(x[choose], y[choose], s=36, facecolors="none", edgecolors=color, linewidths=0.8, label=f"{label} ({int(choose.sum())})", rasterized=True)
    axis.set(title=f"COSMOS{COSMOS} detection image | tile {tile_index:02d} | 7DT-only + matched", xlabel="X pixel", ylabel="Y pixel")
    axis.legend(loc="upper right")
    figure.tight_layout()
    _ = _save_figure(
        figure, f"detection-image tile {tile_index:02d} with 7DT-only and matched sources",
        f"detection_tile_{tile_index:02d}_7dt_only_matched.png",
        "detection_tiles",
    )


Saved detection-image tile 01 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_01_7dt_only_matched.png


Saved detection-image tile 02 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_02_7dt_only_matched.png


Saved detection-image tile 03 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_03_7dt_only_matched.png


Saved detection-image tile 04 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_04_7dt_only_matched.png


Saved detection-image tile 05 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_05_7dt_only_matched.png


...

Saved detection-image tile 16 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_16_7dt_only_matched.png


Saved detection-image tile 17 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_17_7dt_only_matched.png


Saved detection-image tile 18 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_18_7dt_only_matched.png


Saved detection-image tile 19 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_19_7dt_only_matched.png


Saved detection-image tile 20 with 7DT-only and matched sources to /Users/mac_seo/Research_local/7DTonCOSMOS/output/COSMOS1/singlebandphot/matching/figures/detection_tiles/detection_tile_20_7dt_only_matched.png
